# CheXpert CBM Evaluation & Concept Inspection
Run standard, LF-CBM, and VLG-CBM evaluations on the CheXpert test/valid split, then inspect concept contributions.

## 1) Configure paths
Set your run directories and data paths. Use `label_set=chexpert` and choose pathology/competition flags for eval.

In [ ]:
# Paths
DATA_DIR = "/workspace/CheXpert-v1.0-small"
STANDARD_CKPT = "checkpoints/chex_patho/best_model.pth"  # from train.py
LF_RUN_DIR = "checkpoints/lfcbm_chexpert"  # contains W_c.pt, W_g.pt, b_g.pt, proj_mean/std, config.json
VLG_RUN_DIR = "checkpoints/vlg_cbm_chexpert"  # contains backbone/concept_layer/final weights
OUTPUT_DIR = "notebook_eval/chexpert"
LABEL_SET = "chexpert"  # fixed
USE_PATHOLOGY = True
USE_COMP = False
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 8


## 2) Standard model eval (train.py checkpoint)
Uses eval.py to compute AUROC/AP on the chosen split.

In [ ]:
import os, json, subprocess, pathlib
os.makedirs(OUTPUT_DIR, exist_ok=True)

split = "valid"  # or "test" if you have test.csv
cmd = [
    "python", "eval.py",
    "--data_dir", DATA_DIR,
    "--label_set", LABEL_SET,
    "--model", "densenet121",
    "--checkpoint", STANDARD_CKPT,
    "--output", os.path.join(OUTPUT_DIR, f"standard_{split}"),
    "--img_size", str(IMG_SIZE),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--split", split,
]
if USE_PATHOLOGY:
    cmd.append("--pathology_labels")
if USE_COMP:
    cmd.append("--competition_labels")
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

metrics_path = os.path.join(OUTPUT_DIR, f"standard_{split}", f"{split}_metrics.json")
with open(metrics_path) as f:
    std_metrics = json.load(f)
print("Standard model mean AUROC:", std_metrics.get("auroc", {}).get("mean", "N/A"))


## 3) LF-CBM evaluation (use evaluate_nec.py full model)
Computes accuracy for COVID-QU; AUROC/AP for CheXpert.

In [ ]:
split = "valid"
cmd = [
    "python", "evaluate_nec.py",
    "--model_dir", LF_RUN_DIR,
    "--data_dir", DATA_DIR,
    "--label_set", LABEL_SET,
    "--split", split,
    "--batch_size", str(BATCH_SIZE),
    "--nec_levels", "60",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
# Metrics saved to nec_metrics.csv in LF_RUN_DIR


## 4) VLG-CBM evaluation (use evaluate_nec.py full model)
Requires annotations and VLG artifacts in VLG_RUN_DIR.

In [ ]:
split = "valid"
cmd = [
    "python", "evaluate_nec.py",
    "--model_dir", VLG_RUN_DIR,
    "--data_dir", DATA_DIR,
    "--label_set", LABEL_SET,
    "--split", split,
    "--batch_size", str(BATCH_SIZE),
    "--nec_levels", "60",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 5) Concept contributions (LF/VLG)
Load W_g and concepts to view top positive/negative concepts per class.

In [ ]:
import torch, numpy as np

def load_concepts(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

def top_concepts(run_dir, labels, topk=5):
    W_g = torch.load(os.path.join(run_dir, "W_g.pt"))
    concepts = load_concepts(os.path.join(run_dir, "concepts.txt"))
    W_np = W_g.cpu().numpy()
    for cls_idx, cls in enumerate(labels):
        weights = W_np[cls_idx]
        pos_idx = np.argsort(weights)[-topk:][::-1]
        neg_idx = np.argsort(weights)[:topk]
        print(f"\n{cls} (run={run_dir}):")
        print("  Positive:")
        for i in pos_idx:
            print(f"    + {concepts[i]}: {weights[i]:.3f}")
        print("  Negative:")
        for i in neg_idx:
            print(f"    - {concepts[i]}: {weights[i]:.3f}")

# Example: LF run
labels = CHEXPERT_PATHOLOGY_LABELS if USE_PATHOLOGY else (CHEXPERT_COMPETITION_LABELS if USE_COMP else CHEXPERT_PATHOLOGY_LABELS)
top_concepts(LF_RUN_DIR, labels)


## 6) CheXagent sanity check for top concepts (optional)
Fetch CheXagent HF model and print similarities between a sample image and top concepts.

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image

CHEX_MODEL = "StanfordAIMI/CheXagent-8b"
chex_processor = AutoProcessor.from_pretrained(CHEX_MODEL, trust_remote_code=True)
chex_model = AutoModelForCausalLM.from_pretrained(CHEX_MODEL, trust_remote_code=True, torch_dtype=torch.float16).to("cuda").eval()

# Configure a sample image path and top concepts to probe
sample_image = "/path/to/sample_xray.png"
probe_concepts = ["Pleural effusion", "Ground-glass opacity"]

img = Image.open(sample_image).convert("RGB")
inputs = chex_processor(images=[img], return_tensors="pt").to("cuda")
with torch.no_grad():
    vision_out = chex_model.model.vision_model(pixel_values=inputs["pixel_values"])
    vision_feat = vision_out.last_hidden_state.mean(dim=1)
    vision_feat = torch.nn.functional.normalize(vision_feat, dim=1)

# Text embeddings via Mistral not included here; plug in your text encoder to compare.
print("CheXagent vision embedding shape:", vision_feat.shape)
